In [ ]:
import torch
import torchvision
import torch.utils.data
import sklearn.model_selection
import torchvision.transforms.v2

transforms = torchvision.transforms.v2.Compose(
    [
        torchvision.transforms.v2.Resize(224),
        torchvision.transforms.v2.Grayscale(num_output_channels=3),
        torchvision.transforms.v2.ToImage(),
        torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
        torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
    ]
)
train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)

train_idxs, _ = sklearn.model_selection.train_test_split(list(range(len(train_ds))), stratify=train_ds.targets, test_size=0.99, random_state=42)
test_idxs, _ = sklearn.model_selection.train_test_split(list(range(len(test_ds))), stratify=test_ds.targets, test_size=0.99, random_state=42)
train_y = train_ds.targets[train_idxs]
test_y = test_ds.targets[test_idxs]
train_ds = torch.utils.data.Subset(train_ds, indices=train_idxs)
test_ds = torch.utils.data.Subset(test_ds, indices=test_idxs)

len(train_idxs)

In [ ]:
import zigzag.nn
import zigzag.utils
import zigzag.pipelines

PARAMS = [
    zigzag.pipelines.Params(k_neighbors=2, dimension=3),
    zigzag.pipelines.Params(k_neighbors=3, dimension=3),
    zigzag.pipelines.Params(k_neighbors=4, dimension=3),
    zigzag.pipelines.Params(k_neighbors=5, dimension=3),
]

### ViT

In [ ]:
dumper = zigzag.utils.UniversalDumper("zigzag_results/testing/vit_b_16/after")
# dumper.clear()

In [ ]:
pretrained_dumper = dumper.make_subdumper("pretrained")
model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = torch.nn.Identity()

zigzag.pipelines.validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)
zigzag.pipelines.analyze_bulk(model, train_ds, PARAMS, pretrained_dumper, class_labels=train_y)

In [ ]:
finetuned_dumper = dumper.make_subdumper("finetuned")

model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = pretrained_dumper.get_dump("trained_head")

zigzag.pipelines.train_validate(model, train_ds, test_ds, dumper, learning_rate=1e-5)
zigzag.pipelines.analyze_bulk(model, train_ds, PARAMS, finetuned_dumper, class_labels=train_y)

## ResNet

In [ ]:
dumper = zigzag.utils.UniversalDumper("zigzag_results/testing/resnet18/after")
# dumper.clear()

In [ ]:
pretrained_dumper = dumper.make_subdumper("pretrained")
model = torchvision.models.resnet18(
    num_classes=1000, weights=torchvision.models.ResNet18_Weights.DEFAULT
)
model.fc = torch.nn.Identity()

zigzag.pipelines.validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)
zigzag.pipelines.analyze_bulk(model, train_ds, PARAMS, pretrained_dumper) #, class_labels=train_y)

In [ ]:
finetuned_dumper = dumper.make_subdumper("finetuned")

model = torchvision.models.resnet18(
    num_classes=1000, weights=torchvision.models.ResNet18_Weights.DEFAULT
)
model.fc = pretrained_dumper.get_dump("trained_head")

zigzag.pipelines.train_validate(model, train_ds, test_ds, dumper, learning_rate=1e-5)
zigzag.pipelines.analyze_bulk(model, train_ds, PARAMS, finetuned_dumper) #, class_labels=train_y)